# TurboQuant KV-cache compression

TurboQuant is not a weight quantizer; it compresses the runtime KV cache.
The paper reports "absolute quality neutrality with 3.5 bits per channel" for KV-cache quantization.

Sources:
- Paper: https://arxiv.org/abs/2504.19874
- Community PyTorch implementation: https://github.com/tonbistudio/turboquant-pytorch
- Community implementation: https://github.com/OnlyTerp/turboquant

This notebook captures a real Hugging Face `past_key_values` cache, applies a self-contained
TurboQuant-style random-rotation + scalar quantization reference path, saves a cache artifact,
and reports theoretical packed memory and cosine/MSE reconstruction quality.
Use the community repos or future vLLM/llama.cpp integrations for production kernels.

In [1]:
!pip -q install -U transformers accelerate sentencepiece safetensors

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 77.0 MB/s eta 0:00:00


## Cell 2 — Helpers

In [2]:

import json
import math
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM


def bytes_to_mib(n):
    return n / (1024 ** 2)


def make_random_orthogonal(dim, device, dtype, seed=1234):
    g = torch.Generator(device=device)
    g.manual_seed(seed)
    a = torch.randn(dim, dim, generator=g, device=device, dtype=torch.float32)
    q, _ = torch.linalg.qr(a)
    return q.to(device=device, dtype=dtype)


def quantize_rotated_vectors(x, bits, rotation):
    """Apply random rotation then symmetric scalar quantization.""",
    orig_shape = x.shape
    dim = orig_shape[-1]
    flat = x.reshape(-1, dim).to(rotation.dtype)
    rotated = flat @ rotation
    qmax = (2 ** (bits - 1)) - 1
    scale = rotated.abs().amax(dim=-1, keepdim=True).clamp(min=1e-6) / qmax
    q = torch.round(rotated / scale).clamp(-qmax, qmax).to(torch.int8)
    deq = (q.to(rotation.dtype) * scale) @ rotation.T
    return {
        "q": q.cpu(),
        "scale": scale.cpu().to(torch.float16),
        "dequantized": deq.reshape(orig_shape).to(x.dtype),
    }


def iter_kv_layers(past_key_values):
    """Iterate over (key, value) tensors regardless of whether past_key_values
    is the old list-of-tuples format or a newer transformers DynamicCache object."""
    if past_key_values is None:
        return
    try:
        # DynamicCache: has .key_cache / .value_cache as lists
        key_cache = past_key_values.key_cache
        value_cache = past_key_values.value_cache
        for k, v in zip(key_cache, value_cache):
            yield k, v
    except AttributeError:
        # Legacy format: list of (key, value) tuples
        for layer in past_key_values:
            if isinstance(layer, (tuple, list)):
                yield layer[0], layer[1]


def kv_cache_bytes(past_key_values):
    total = 0
    for key, value in iter_kv_layers(past_key_values):
        total += key.numel() * key.element_size()
        total += value.numel() * value.element_size()
    return total


def cosine_report(original, reconstructed):
    a = original.reshape(-1, original.shape[-1]).float()
    b = reconstructed.reshape(-1, reconstructed.shape[-1]).float()
    cos = F.cosine_similarity(a, b, dim=-1)
    mse = F.mse_loss(b, a).item()
    return float(cos.mean().item()), float(cos.min().item()), float(mse)


def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(json.dumps(payload, indent=2))


## Cell 3 — Config, Model Load, Compress & Save

In [3]:
MODEL_ID = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = Path("/content/quant_outputs/turboquant_kv")
# Long prompt to create a sizeable KV cache for meaningful compression numbers
PROMPT = "TurboQuant compresses the key value cache for long context inference. " * 32
BITS = 3
INCLUDE_QJL_RESIDUAL_BITS = True   # adds 1 residual sign bit per value (from QJL paper)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Device: {device}  |  Model: {MODEL_ID}  |  Bits: {BITS}")

t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype).to(device)
model.eval()
load_seconds = time.perf_counter() - t0
print(f"Model loaded in {load_seconds:.1f}s")

# ── Prefill to capture real KV cache ────────────────────────────────────────
inputs = tokenizer(PROMPT, return_tensors="pt").to(device)
print(f"Prompt tokens: {inputs['input_ids'].shape[-1]}")
with torch.no_grad():
    t0 = time.perf_counter()
    outputs = model(**inputs, use_cache=True)
    prefill_seconds = time.perf_counter() - t0
past = outputs.past_key_values
print(f"Prefill done in {prefill_seconds:.2f}s")

head_dim = next(iter_kv_layers(past))[0].shape[-1]
rotation = make_random_orthogonal(head_dim, device, dtype)

artifact = {"method": "turboquant_reference", "bits": BITS, "layers": []}
reports = []
packed_bits = 0

t0 = time.perf_counter()
for layer_idx, (key, value) in enumerate(iter_kv_layers(past)):
    qk = quantize_rotated_vectors(key, BITS, rotation)
    qv = quantize_rotated_vectors(value, BITS, rotation)
    k_mean, k_min, k_mse = cosine_report(key, qk["dequantized"].to(device))
    v_mean, v_min, v_mse = cosine_report(value, qv["dequantized"].to(device))
    n_values  = key.numel() + value.numel()
    n_vectors = key.reshape(-1, head_dim).shape[0] + value.reshape(-1, head_dim).shape[0]
    packed_bits += n_values * BITS
    packed_bits += n_vectors * 16  # per-vector fp16 scale
    if INCLUDE_QJL_RESIDUAL_BITS:
        packed_bits += n_values  # one residual sign bit per coordinate
    artifact["layers"].append({
        "key_q":   qk["q"],
        "key_scale": qk["scale"],
        "value_q": qv["q"],
        "value_scale": qv["scale"],
    })
    reports.append({
        "layer": layer_idx,
        "key_cosine_mean":   k_mean, "key_cosine_min":   k_min, "key_mse":   k_mse,
        "value_cosine_mean": v_mean, "value_cosine_min": v_min, "value_mse": v_mse,
    })
quant_seconds = time.perf_counter() - t0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(artifact, OUTPUT_DIR / "compressed_kv_cache.pt")

fp_bytes     = kv_cache_bytes(past)
packed_bytes = math.ceil(packed_bits / 8)
metrics = {
    "method":                        "turboquant_reference_kv_cache",
    "model_id":                      MODEL_ID,
    "bits":                          BITS,
    "qjl_residual_bits_included":    INCLUDE_QJL_RESIDUAL_BITS,
    "prompt_tokens":                 int(inputs["input_ids"].shape[-1]),
    "num_layers":                    len(reports),
    "head_dim":                      head_dim,
    "fp_cache_mib":                  bytes_to_mib(fp_bytes),
    "theoretical_packed_cache_mib":  bytes_to_mib(packed_bytes),
    "compression_ratio":             fp_bytes / packed_bytes,
    "prefill_seconds":               round(prefill_seconds, 3),
    "quant_seconds":                 round(quant_seconds, 3),
    "artifact_path":                 str(OUTPUT_DIR / "compressed_kv_cache.pt"),
    "reports":                       reports,
}
save_json(OUTPUT_DIR / "metrics.json", metrics)

Device: cuda  |  Model: TinyLlama/TinyLlama-1.1B-Chat-v1.0  |  Bits: 3


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded in 44.7s
Prompt tokens: 450
Prefill done in 0.83s
{
  "method": "turboquant_reference_kv_cache",
  "model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
  "bits": 3,
  "qjl_residual_bits_included": true,
  "prompt_tokens": 450,
  "num_layers": 22,
  "head_dim": 64,
  "fp_cache_mib": 9.66796875,
  "theoretical_packed_cache_mib": 2.56805419921875,
  "compression_ratio": 3.764705882352941,
  "prefill_seconds": 0.832,
  "quant_seconds": 0.228,
  "artifact_path": "/content/quant_outputs/turboquant_kv/compressed_kv_cache.pt",
  "reports": [
    {
      "layer": 0,
      "key_cosine_mean": 0.9705765843391418,
      "key_cosine_min": 0.9164609313011169,
      "key_mse": 0.08637145161628723,
      "value_cosine_mean": 0.9689784049987793,
      "value_cosine_min": 0.9438118934631348,
      "value_mse": 9.400514500157442e-06
    },
    {
      "layer": 1,
      "key_cosine_mean": 0.9700478911399841,
      "key_cosine_min": 0.9263651967048645,
      "key_mse": 0.3495681583881378,
      "v

## Cell 4 — (Optional) Clone Community Implementation

For production kernels and deeper experiments, clone one of the community implementations.

In [4]:
# !git clone https://github.com/tonbistudio/turboquant-pytorch /content/turboquant-pytorch
# %cd /content/turboquant-pytorch
# !pip -q install -r requirements.txt
# !python validate.py

In [5]:
import shutil
from google.colab import files

# Zip the entire quant_outputs directory
shutil.make_archive("/content/quant_outputs", 'zip', "/content/quant_outputs")

# Download the zip file
files.download("/content/quant_outputs.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>